# 03 — Matched-flow join (Arm B)

Builds the subset of flows present in **both** versions, so that label is the only
thing that differs.

Design facts established by `03b_diagnose` on real data:

- **Original timestamps:** minute resolution, 12-hour clock with no AM/PM marker.
- **Clock offset:** the improved release runs **+3h ahead** of the original
  (original is Atlantic local time; the regenerated release is very likely UTC).
  Monday minute-sets overlap 487/487 at +3h — a perfect triangle peak.
- **Flow direction is inconsistent in the original** (8,239 distinct src IPs on
  Monday vs 90 in the improved; server→client rows exist). The join key therefore
  uses **canonical endpoint ordering** — the two `(ip:port)` endpoints sorted —
  so a flow matches its counterpart regardless of recorded direction.
- Validated forward-only match rate at +3h: 77.6% of Monday. Canonical ordering
  should recover part of the remainder.

The shift is still found by search, not hard-coded — if a future data refresh
changes the offset, the notebook adapts and logs it. Everything lands in
`results/match_calibration.json` and `PROTOCOL_AMENDMENTS.md`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'
sys.path.insert(0, os.path.join(DRIVE_ROOT, 'src'))

import importlib, config as C, helpers as H
importlib.reload(C); importlib.reload(H)

import pandas as pd, numpy as np, json

orig = pd.read_parquet(os.path.join(C.INTERIM, 'original.parquet'))
impr = pd.read_parquet(os.path.join(C.INTERIM, 'improved.parquet'))
print(f'original: {len(orig):,}   improved: {len(impr):,}')
assert len(orig) == 2_830_743, 'Re-run notebook 02 — the cached original is unrepaired.'

Mounted at /content/drive
original: 2,830,743   improved: 2,099,976


In [2]:
# --- 1. Parse validation (capture week: 3-7 July 2017 => month must be 7) ---
# CRITICAL: format='mixed' parses each element individually. The original
# release mixes timestamp formats ACROSS FILES (some carry seconds, some do
# not). Pandas' default whole-column format inference locks onto the first
# file's pattern and silently NaTs every file written differently — in this
# dataset that wiped out all of Monday (18.72% NaT == Monday's exact share).

def parse_mixed(series, dayfirst):
    return pd.to_datetime(series, errors='coerce', dayfirst=dayfirst, format='mixed')

def choose_parse(series, name, day=None):
    a = parse_mixed(series, True)
    va = float((a.dt.month == 7).mean())
    if va >= 0.999:
        ts, mode, v = a, 'dayfirst=True', va
    else:
        b = parse_mixed(series, False)
        vb = float((b.dt.month == 7).mean())
        ts, mode, v = (a, 'dayfirst=True', va) if va >= vb else (b, 'dayfirst=False', vb)
    nat = float(ts.isna().mean())
    print(f'{name:9s} {mode:16s} month==7: {v:6.1%}   NaT: {nat:.2%}')
    if day is not None and (nat > 0.001 or v < 0.999):
        print(f'  per-day NaT rates for {name}:')
        print((ts.isna().groupby(day).mean() * 100).round(2).to_string())
    assert v > 0.999 and nat < 0.001, (
        f'{name}: timestamp parse failed validation — a file-level format quirk '
        f'slipped through. Inspect the per-day table above before continuing.')
    return ts, {'mode': mode, 'valid_july': v, 'nat_rate': nat}

calib = {}
ts_o, calib['parse_original'] = choose_parse(orig['timestamp'], 'original', orig['day'])
ts_i, calib['parse_improved'] = choose_parse(impr['timestamp'], 'improved', impr['day'])

original  dayfirst=True    month==7: 100.0%   NaT: 0.00%
improved  dayfirst=True    month==7: 100.0%   NaT: 0.00%


In [3]:
# --- 2. Clock repair (12h signature: mass at hours 1-7, none at 13-17) ------
def apply_12h_repair(ts, name):
    h = ts.dt.hour
    early, late = float(h.between(1, 7).mean()), float(h.between(13, 17).mean())
    fix = early > 0.02 and late < 0.005
    print(f'{name:9s} hours1-7 {early:6.2%}  hours13-17 {late:6.2%}  12h-signature {fix}')
    if fix:
        amb = h.between(1, 7)
        ts = ts + pd.to_timedelta(amb.astype('int64') * 12, unit='h')
        print(f'          +12h applied to {int(amb.sum()):,} rows')
        return ts, int(amb.sum())
    return ts, 0

ts_o, calib['clock_fix_original'] = apply_12h_repair(ts_o, 'original')
ts_i, calib['clock_fix_improved'] = apply_12h_repair(ts_i, 'improved')

print('\nhour ranges after repair:')
print('original:', int(ts_o.dt.hour.min()), '-', int(ts_o.dt.hour.max()))
print('improved:', int(ts_i.dt.hour.min()), '-', int(ts_i.dt.hour.max()))

original  hours1-7 51.99%  hours13-17  0.00%  12h-signature True
          +12h applied to 1,471,716 rows
improved  hours1-7  0.00%  hours13-17 59.14%  12h-signature False

hour ranges after repair:
original: 8 - 17
improved: 11 - 20


In [4]:
# --- 3. Canonical endpoint key ----------------------------------------------
# The original records some flows server->client, so (src, dst) is unreliable.
# Sort the two endpoints; a flow then matches its counterpart in either direction.

def keyframe(df, ts, shift_h=0):
    ip_s = df['src_ip'].astype(str).str.strip()
    ip_d = df['dst_ip'].astype(str).str.strip()
    po_s = pd.to_numeric(df['src_port'], errors='coerce').astype('Int64').astype(str)
    po_d = pd.to_numeric(df['dst_port'], errors='coerce').astype('Int64').astype(str)
    ep_a = (ip_s + ':' + po_s).values
    ep_b = (ip_d + ':' + po_d).values
    first = ep_a <= ep_b
    k = pd.DataFrame({
        'k_ep1': np.where(first, ep_a, ep_b),
        'k_ep2': np.where(first, ep_b, ep_a),
        'k_protocol': pd.to_numeric(df['protocol'], errors='coerce')
                        .astype('Int64').astype(str),
    }, index=df.index)
    k['minute'] = (ts + pd.Timedelta(hours=shift_h)).dt.floor('min')
    return k

KEY = ['k_ep1', 'k_ep2', 'k_protocol', 'minute']

In [5]:
# --- 4. Shift calibration on Monday ------------------------------------------
om = orig['day'] == 'monday'
im = impr['day'] == 'monday'
ki = keyframe(impr.loc[im], ts_i.loc[im])
iset = set(map(tuple, ki.astype(str).values))

results = {}
for shift in range(-12, 13):
    ko = keyframe(orig.loc[om], ts_o.loc[om], shift)
    n = sum(1 for t in map(tuple, ko.astype(str).values) if t in iset)
    results[shift] = n
    print(f'shift {shift:+3d}h   canonical-key matches on Monday: {n:>9,}')

best = max(results, key=results.get)
n_monday = int(om.sum())
print(f'\nbest shift: {best:+d}h with {results[best]:,} matches '
      f'({results[best]/n_monday:.1%} of Monday)')

calib['shift_search'] = {str(k): v for k, v in results.items()}
calib['shift_applied_hours'] = best

SEARCH_OK = results[best] >= 0.05 * n_monday
if SEARCH_OK:
    if best != 0:
        ts_o = ts_o + pd.Timedelta(hours=best)
        print(f'applied {best:+d}h to the original clock')
else:
    print('\n' + '=' * 70)
    print('SEARCH FAILED — under 5% at every shift. The NEXT CELL prints the')
    print('diagnostics automatically. Let it run, send its output.')
    print('=' * 70)

shift -12h   canonical-key matches on Monday:         0
shift -11h   canonical-key matches on Monday:         0
shift -10h   canonical-key matches on Monday:         0
shift  -9h   canonical-key matches on Monday:         0
shift  -8h   canonical-key matches on Monday:         0
shift  -7h   canonical-key matches on Monday:         0
shift  -6h   canonical-key matches on Monday:         0
shift  -5h   canonical-key matches on Monday:        10
shift  -4h   canonical-key matches on Monday:       749
shift  -3h   canonical-key matches on Monday:     1,567
shift  -2h   canonical-key matches on Monday:     2,118
shift  -1h   canonical-key matches on Monday:     2,614
shift  +0h   canonical-key matches on Monday:     3,077
shift  +1h   canonical-key matches on Monday:     3,804
shift  +2h   canonical-key matches on Monday:     5,036
shift  +3h   canonical-key matches on Monday:   489,468
shift  +4h   canonical-key matches on Monday:     5,036
shift  +5h   canonical-key matches on Monday:   

In [6]:
# --- 4b. PROBE (runs automatically) ------------------------------------------
# Prints real key strings from both sides and tests each component in isolation.

PSHIFT = 3   # the shift 03b validated

ko = keyframe(orig.loc[om], ts_o.loc[om], PSHIFT).astype(str)
ki2 = keyframe(impr.loc[im], ts_i.loc[im]).astype(str)

print('ORIGINAL keys (3 rows, shift +%dh):' % PSHIFT)
for _, r in ko.head(3).iterrows():
    print('  ', tuple(r))
print('IMPROVED keys (3 rows):')
for _, r in ki2.head(3).iterrows():
    print('  ', tuple(r))

print('\nminute strings, repr (hidden precision shows here):')
print('  original:', repr(ko['minute'].iloc[0]))
print('  improved:', repr(ki2['minute'].iloc[0]))
print('protocol strings, value counts:')
print('  original:', ko['k_protocol'].value_counts().head(5).to_dict())
print('  improved:', ki2['k_protocol'].value_counts().head(5).to_dict())

# piecewise membership on a 100k sample of the original
osamp = ko.head(100_000)
i_ep  = set(map(tuple, ki2[['k_ep1', 'k_ep2']].values))
i_min = set(ki2['minute'])
i_full = set(map(tuple, ki2.values))

ep_hits   = sum(1 for t in map(tuple, osamp[['k_ep1', 'k_ep2']].values) if t in i_ep)
min_hits  = sum(1 for m in osamp['minute'] if m in i_min)
full_hits = sum(1 for t in map(tuple, osamp.values) if t in i_full)
print(f'\npiecewise hits on {len(osamp):,} original rows at +{PSHIFT}h:')
print(f'  endpoint-pair only : {ep_hits:>8,}  ({ep_hits/len(osamp):.1%})')
print(f'  minute only        : {min_hits:>8,}  ({min_hits/len(osamp):.1%})')
print(f'  full canonical key : {full_hits:>8,}  ({full_hits/len(osamp):.1%})')

# bridge to 03b: the forward directed 5-tuple key that matched 410,996 there
def fwd(df, ts, shift_h=0):
    k = df[['src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol']].copy()
    for c in ['src_ip', 'dst_ip']:
        k[c] = k[c].astype(str).str.strip()
    for c in ['src_port', 'dst_port', 'protocol']:
        k[c] = pd.to_numeric(k[c], errors='coerce').astype('Int64').astype(str)
    k['minute'] = (ts + pd.Timedelta(hours=shift_h)).dt.floor('min').astype(str)
    return k

fi = set(map(tuple, fwd(impr.loc[im], ts_i.loc[im]).astype(str).values))
fo = fwd(orig.loc[om], ts_o.loc[om], PSHIFT).astype(str)
fhits = sum(1 for t in map(tuple, fo.values) if t in fi)
print(f'\nforward 5-tuple key at +{PSHIFT}h (03b reference was 410,996): {fhits:,}')

print('\nREADING:')
print('  forward ~410k but canonical ~0    -> endpoint construction bug, strings above show it')
print('  forward also ~0                   -> ts_o/ts_i differ from 03b: compare hour ranges')
print('  minute-only ~0                    -> minute string format mismatch (see reprs)')

ORIGINAL keys (3 rows, shift +3h):
   ('192.168.10.5:49188', '8.254.250.126:80', '6', '2017-07-03 14:55:00')
   ('192.168.10.5:49188', '8.254.250.126:80', '6', '2017-07-03 14:55:00')
   ('192.168.10.5:49188', '8.254.250.126:80', '6', '2017-07-03 14:55:00')
IMPROVED keys (3 rows):
   ('8.0.6.4:0', '8.6.0.1:0', '0', '2017-07-03 11:56:00')
   ('192.168.10.3:123', '192.168.10.9:123', '17', '2017-07-03 11:56:00')
   ('192.168.10.12:5353', '224.0.0.251:5353', '17', '2017-07-03 11:57:00')

minute strings, repr (hidden precision shows here):
  original: '2017-07-03 14:55:00'
  improved: '2017-07-03 11:56:00'
protocol strings, value counts:
  original: {'6': 305423, '17': 224178, '0': 317}
  improved: {'17': 224023, '6': 147204, '0': 305, '1': 92}

piecewise hits on 100,000 original rows at +3h:
  endpoint-pair only :   99,999  (100.0%)
  minute only        :  100,000  (100.0%)
  full canonical key :    1,190  (1.2%)

forward 5-tuple key at +3h (03b reference was 410,996): 3,077

READING:
  for

In [7]:
# --- 5. Join at minute resolution --------------------------------------------
assert SEARCH_OK, ('Shift search failed — fix the key first. '
                   'The probe cell above shows where it breaks.')

def with_key(df, ts):
    df = df.copy()
    k = keyframe(df, ts)
    for c in KEY:
        df[c] = k[c]
    return df

orig_k = with_key(orig, ts_o)
impr_k = with_key(impr, ts_i)

def unique_keyed(df, name):
    n0 = len(df)
    df = df.dropna(subset=['minute'])
    n1 = len(df)
    counts = df.groupby(KEY, dropna=False).size()
    dup_keys = counts[counts > 1].index
    df = df.set_index(KEY)
    df = df[~df.index.isin(dup_keys)].reset_index()
    n2 = len(df)
    print(f'{name}: {n0:,} -> {n1:,} (valid minute) -> {n2:,} (unique key)  '
          f'[{n2/n0:.1%} retained]')
    return df, {'rows': n0, 'valid_minute': n1, 'unique_key': n2}

orig_u, calib['unique_original'] = unique_keyed(orig_k, 'original')
impr_u, calib['unique_improved'] = unique_keyed(impr_k, 'improved')

extra = ['label'] + (['attempted'] if 'attempted' in impr_u.columns else [])
matched = orig_u.merge(impr_u[KEY + extra], on=KEY, how='inner',
                       suffixes=('', '_improved'))
if 'label_improved' not in matched.columns and 'label_y' in matched.columns:
    matched = matched.rename(columns={'label_y': 'label_improved',
                                      'label_x': 'label'})
matched = matched.rename(columns={'label': 'label_original'})

match_rate = len(matched) / len(orig_u)
calib['matched_flows'] = int(len(matched))
calib['match_rate_vs_unique_original'] = round(match_rate, 4)
calib['match_rate_vs_all_original'] = round(len(matched) / len(orig), 4)

print(f'\nmatched flows: {len(matched):,}')
print(f'match rate vs uniquely-keyed original: {match_rate:.2%}')
print(f'match rate vs all original flows:      {len(matched)/len(orig):.2%}')

gate_g2 = match_rate >= C.MIN_MATCH_RATE
calib['gate_g2_pass'] = bool(gate_g2)
print('\nGATE G2:', 'PASS' if gate_g2 else 'FAIL')

if len(matched) == 0:
    raise RuntimeError('Zero matched flows — run 03b_diagnose.ipynb before '
                       'touching anything below.')
if not gate_g2:
    print('Arm B is not viable at this match rate. Reframe as an incomparability')
    print('result and report the rate itself. Do not lower the threshold.')

original: 2,830,743 -> 2,830,743 (valid minute) -> 1,767,843 (unique key)  [62.5% retained]
improved: 2,099,976 -> 2,099,976 (valid minute) -> 2,073,720 (unique key)  [98.7% retained]

matched flows: 1,604,995
match rate vs uniquely-keyed original: 90.79%
match rate vs all original flows:      56.70%

GATE G2: PASS


In [8]:
# --- 6. Log calibration + amend the protocol ---------------------------------
with open(os.path.join(C.RESULTS, 'match_calibration.json'), 'w') as f:
    json.dump(calib, f, indent=2, default=str)
print('saved results/match_calibration.json')

from datetime import date
amendment = f'''
## Amendment — {date.today().isoformat()} — Arm B join key

Protocol section 3 specified a directed 5-tuple key rounded to the second.
Diagnostics on the real data (03b) established three format facts, all fixed
before any label comparison was computed:

1. The original release MIXES timestamp formats across its own files (some
   files carry seconds, some do not). Whole-column format inference in pandas
   therefore silently voided entire files — 18.72% NaT, exactly Monday's share
   of the corpus — until per-element parsing (format='mixed') was used. It also
   records timestamps at MINUTE resolution in several files, on a 12-hour clock
   with no AM/PM marker; hours 1-7 were repaired to PM
   ({calib.get('clock_fix_original', 0):,} rows).
2. The improved release's clock runs {calib.get('shift_applied_hours', 0):+d}h
   relative to the original (minute-sets overlap perfectly at that shift;
   original is Atlantic local time, the regenerated release is consistent with
   UTC). The shift was found by search and applied to the original.
3. The original's flow direction is inconsistent (server->client rows exist;
   8,239 distinct Monday src IPs vs 90 in the improved). The key therefore uses
   canonically ordered endpoints, matching flows regardless of direction.

Join key: (sorted endpoint pair, protocol, minute). Ambiguous keys (repeated
within a version) are dropped and the retention reported.
'''
with open(os.path.join(C.DRIVE_ROOT, 'PROTOCOL_AMENDMENTS.md'), 'a') as f:
    f.write(amendment)
print('appended to PROTOCOL_AMENDMENTS.md')

saved results/match_calibration.json
appended to PROTOCOL_AMENDMENTS.md


In [9]:
# --- 7. THE core artefact: label movement on identical flows ------------------
mo = H.coarse_class(matched['label_original'])
mi = H.coarse_class(matched['label_improved'])

xtab = pd.crosstab(mo, mi, rownames=['original'], colnames=['improved'])
xtab.to_csv(os.path.join(C.RESULTS, 'table5_label_transition_matrix.csv'))
print('saved table5_label_transition_matrix.csv')

changed = (matched['label_original'].astype(str).str.strip().str.upper()
           != matched['label_improved'].astype(str).str.strip().str.upper())
print(f'\nlabels changed on matched flows: '
      f'{changed.sum():,} / {len(matched):,} = {changed.mean():.2%}')
xtab

saved table5_label_transition_matrix.csv

labels changed on matched flows: 70,552 / 1,604,995 = 4.40%


improved,BENIGN,Bot,BruteForce,DDoS,DoS,Heartbleed,Infiltration,PortScan,WebAttack
original,,,,,,,,,
BENIGN,1253431,166,1,7476,441,0,56879,31,7
Bot,2,5,0,0,0,0,0,0,0
BruteForce,0,0,1218,0,0,0,0,0,0
DDoS,6,0,0,21241,0,0,0,0,0
DoS,1,0,0,0,105330,0,0,0,0
Heartbleed,0,0,0,0,0,10,0,0,0
Infiltration,0,0,0,0,0,0,32,0,0
PortScan,0,0,0,0,0,0,0,158476,0
WebAttack,0,0,0,0,0,0,0,0,242


In [10]:
# --- 8. Direction of change + per-class relabel rates ------------------------
bo = H.binarise(matched['label_original'])
bi = H.binarise(matched['label_improved'])

summary = pd.DataFrame([{
    'matched_flows': len(matched),
    'label_changed': int(changed.sum()),
    'change_rate_pct': round(changed.mean() * 100, 3),
    'benign_to_attack': int(((bo == 0) & (bi == 1)).sum()),
    'attack_to_benign': int(((bo == 1) & (bi == 0)).sum()),
    'attack_family_changed': int(((bo == 1) & (bi == 1) & changed).sum()),
    'attack_to_attempted': int(((bo == 1) & H.is_attempted(matched['label_improved'])).sum()),
    'match_rate_pct': round(match_rate * 100, 3),
}])
H.save_table(summary, 'table6_match_summary.csv')
display(summary.T)

per_class = []
for cls in sorted(set(mo.unique())):
    sel = (mo == cls)
    per_class.append({
        'class': cls,
        'n_matched': int(sel.sum()),
        'n_relabelled': int((sel & changed).sum()),
        'relabel_rate_pct': round(((sel & changed).sum() / max(sel.sum(), 1)) * 100, 3),
    })
pc = pd.DataFrame(per_class).sort_values('n_matched', ascending=False)
H.save_table(pc, 'table7_relabel_rate_per_class.csv')

STABLE = pc.loc[(pc['relabel_rate_pct'] < 0.1) & (pc['n_matched'] >= 1000),
                'class'].tolist()
with open(os.path.join(C.RESULTS, 'h4_control_classes.json'), 'w') as f:
    json.dump(STABLE, f, indent=2)
print('\nH4 control classes (untouched, n>=1000):', STABLE)
pc

saved /content/drive/MyDrive/research/ids-label-correction/results/table6_match_summary.csv (1, 8)


,0
matched_flows,1604995.000
label_changed,70552.000
change_rate_pct,4.396
benign_to_attack,65001.000
attack_to_benign,9.000
attack_family_changed,5542.000
attack_to_attempted,5482.000
match_rate_pct,90.788


saved /content/drive/MyDrive/research/ids-label-correction/results/table7_relabel_rate_per_class.csv (9, 4)

H4 control classes (untouched, n>=1000): ['PortScan', 'DDoS']


,class,n_matched,n_relabelled,relabel_rate_pct
0,BENIGN,1318432,65001,4.930
7,PortScan,158476,0,0.000
4,DoS,105331,5284,5.017
3,DDoS,21247,6,0.028
2,BruteForce,1218,11,0.903
8,WebAttack,242,242,100.000
6,Infiltration,32,1,3.125
5,Heartbleed,10,0,0.000
1,Bot,7,7,100.000


In [11]:
# --- 9. Cache (key columns stripped — they are not features) ------------------
drop_cols = [c for c in ['k_ep1', 'k_ep2', 'k_protocol', 'minute']
             if c in matched.columns]
matched.drop(columns=drop_cols).to_parquet(
    os.path.join(C.INTERIM, 'matched.parquet'), index=False)
print('cached matched subset:', matched.shape[0], 'rows')

cached matched subset: 1604995 rows


## Read the transition matrix carefully

The off-diagonal cells are the paper. Given Table 1, expect the drama in the
Infiltration column and the benign row. A match rate around 70-80% is the honest
outcome — the improved release rebuilt flow boundaries, so some flows genuinely
have no counterpart. Report the rate; do not chase it.

Next: `04_train.ipynb`.